# Baseline models and validation results (BD-01 to BD-05)

**Outcome:** the historical arraignment decision to set bail or remand, not actual detention, future behavior, or a recommended decision.

**Design:** NYC main sample; train and tune on the configured training years, select models on the configured validation year. Three baseline models are evaluated on all three nested main feature sets. CASE_PLUS_GENDER is a separate supplementary diagnostic for each baseline; race and ethnicity are never predictors. The two-field charge rule is included once under CASE.

The notebook reads only development-year partitions if analytic.parquet is available. Otherwise it rebuilds those years in memory using the team's unchanged sample code and pinned raw files. It never reads the test-year raw file, writes analytic.parquet or sample_flow.csv, or runs the final test. Predictions and fitted models are local-only artifacts, excluded from Git submissions.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd

sys.path.append("..")
from src import config, features
from src.evaluate import comparison_table, metrics
from src.models import PREDICTION_COLUMNS, run_baseline_suite, split_by_year
from src.sample import build_sample, load_raw

assert config.ROOT == Path.cwd().resolve().parent, "Run from the repository notebooks directory."
print("Python:", sys.version.split()[0])
print({p: version(p) for p in ["pandas", "numpy", "scikit-learn", "pyarrow"]})

Python: 3.11.18
{'pandas': '3.1.1', 'numpy': '2.4.6', 'scikit-learn': '1.9.1', 'pyarrow': '26.0.1'}


In [2]:
development_years = [*config.TRAIN_YEARS, config.VALID_YEAR]
assert config.TEST_YEAR not in development_years
published = pd.read_csv(config.SAMPLE_FLOW_FILE).set_index("step")
columns = list(dict.fromkeys([
    config.ID_COL, config.YEAR_COL, config.SPLIT_COL, config.MAIN_SAMPLE_COL,
    config.TARGET_Q1, *features.ALL_FEATURES,
]))
if config.ANALYTIC_FILE.is_file():
    development = pd.read_parquet(
        config.ANALYTIC_FILE, columns=columns,
        filters=[(config.YEAR_COL, "in", development_years)],
    )
    source = "analytic.parquet (development-year partitions only)"
else:
    raw = load_raw(years=development_years)
    assert set(raw[config.YEAR_COL].dropna().unique()) == set(development_years)
    development, flow = build_sample(raw)
    del raw
    pd.testing.assert_frame_equal(
        flow.set_index("step")[[str(y) for y in development_years]],
        published[[str(y) for y in development_years]],
        check_dtype=False,
    )
    development = development[columns]
    source = "development years rebuilt with published sample.py; every flow step matched"
assert set(development[config.YEAR_COL].dropna().unique()) == set(development_years)
main = development.loc[development[config.MAIN_SAMPLE_COL]].copy()
del development
expected_counts = {year: int(published.loc["MAIN", str(year)]) for year in development_years}
counts = main.groupby(config.YEAR_COL).size().to_dict()
assert counts == expected_counts, (counts, expected_counts)
parts = split_by_year(main)
assert parts["test"].empty
print("Input:", source)
print("Main cases by year:", counts)
print("Cases by split:", {k: len(v) for k, v in parts.items() if k != "test"})
for fs, cols in features.ALL_SETS.items():
    features.check_leakage(cols, allow=config.SUPPLEMENTARY_ALLOWED.get(fs, ()))
    print(fs, "raw features:", len(cols))

Input: analytic.parquet (development-year partitions only)
Main cases by year: {np.int64(2021): 89189, np.int64(2022): 104007, np.int64(2023): 120457}
Cases by split: {'train': 193196, 'valid': 120457}
CASE raw features: 18
CASE_COURT raw features: 19
CASE_COURT_JUDGE raw features: 20
CASE_PLUS_GENDER raw features: 19


## Training and tuning

The full training sample is used for every grid search; no tuning subsample, class resampling, or class weighting is used. Each model uses the handoff's predefined small grid and the same shuffled, stratified five-fold CV with the configured random seed. Numeric imputation, one-hot categories and (for logistic models) scaling are fitted inside each training fold. The tree is not scaled. Numerical criminal-record gaps are filled with zero for computation while retaining dcjs_missing; this does not interpret a missing record as a true zero. Any residual missing age is also filled with zero under the published BD-01 specification, a limitation to revisit only through the team's feature/preprocessing change process.

Judge grouping is the upstream, outcome-free training-year caseload mapping. Selection uses judge_id, never judge_id_final. Unseen categories have no fitted one-hot column and are ignored; the other observed features still contribute. A previously unseen judge is not assigned a learned judge effect.

Training OOF predictions reuse hyperparameters chosen on these training data; they are a diagnostic rather than a nested-CV performance estimate. The independent validation year is the primary comparison. Nonconverged training fits raise an error rather than silently entering the report.

In [3]:
pred_df, fit_diagnostics, tuning_results = run_baseline_suite(main)
assert pred_df.columns.tolist() == PREDICTION_COLUMNS
assert set(pred_df["split"]) == {"train_oof", "valid"}
assert not pred_df[config.YEAR_COL].eq(config.TEST_YEAR).any()
expected_pairs = {
    (model, fs) for model in ["logit", "lasso", "tree"] for fs in features.ALL_SETS
} | {("rule", "CASE")}
assert set(zip(pred_df["model"], pred_df["feature_set"])) == expected_pairs
for _, rows in pred_df.groupby(["model", "feature_set"], sort=False):
    for split, ref in [("train_oof", parts["train"]), ("valid", parts["valid"])]:
        got = rows.loc[rows["split"].eq(split)].set_index(config.ID_COL).sort_index()
        expected = ref.set_index(config.ID_COL).sort_index()
        assert got.index.equals(expected.index)
        np.testing.assert_array_equal(got["y_true"], expected[config.TARGET_Q1])
assert len(pred_df) == len(expected_pairs) * len(main)
reloaded = pd.read_parquet(config.PRED_BASELINE_FILE)
pd.testing.assert_frame_equal(reloaded, pred_df)
print("Local prediction file:", config.PRED_BASELINE_FILE.name)
print("Prediction rows:", len(pred_df), "; model-feature combinations:", len(expected_pairs))
print("All combinations have identical case coverage and aligned labels.")

Fitting logit / CASE


  selected {'clf__C': 1}; CV AUC 0.936562


Fitting logit / CASE_COURT


  selected {'clf__C': 10}; CV AUC 0.936919


Fitting logit / CASE_COURT_JUDGE


  selected {'clf__C': 1}; CV AUC 0.939821


... (15 outputs left out)

  selected {'clf__max_depth': 12, 'clf__min_samples_leaf': 200}; CV AUC 0.936377


Fitting tree / CASE_PLUS_GENDER


  selected {'clf__max_depth': 12, 'clf__min_samples_leaf': 200}; CV AUC 0.936514


Local prediction file: predictions_baseline.parquet
Prediction rows: 4077489 ; model-feature combinations: 13
All combinations have identical case coverage and aligned labels.


In [4]:
summary = pd.DataFrame([
    {"model": model, "feature_set": fs, "split": split, **metrics(g["y_true"], g["y_prob"])}
    for (model, fs, split), g in pred_df.groupby(["model", "feature_set", "split"], sort=False)
])
config.REPORTS.mkdir(parents=True, exist_ok=True)
summary.to_csv(config.REPORTS / "bd05_metrics.csv", index=False)
fit_diagnostics.to_csv(config.REPORTS / "bd05_fit_diagnostics.csv", index=False)
tuning_results.to_csv(config.REPORTS / "bd05_tuning.csv", index=False)
print("Validation results (AUC / AP higher is better; Brier lower is better):")
print(summary.loc[summary["split"].eq("valid")].to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("Training OOF diagnostics:")
print(summary.loc[summary["split"].eq("train_oof")].to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("Tuning and actual training matrix sizes:")
print(fit_diagnostics.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("Main nested comparison:")
print(comparison_table(pred_df.loc[pred_df["feature_set"].isin(config.FEATURE_SETS)]).to_string())
assert not summary.loc[summary["split"].eq("valid"), "roc_auc"].gt(0.95).any(), "Review possible leakage before interpreting a high AUC."

Validation results (AUC / AP higher is better; Brier lower is better):
model      feature_set split  roc_auc  avg_precision    brier      n  base_rate
 rule             CASE valid 0.846492       0.478021 0.080927 120457   0.128179
logit             CASE valid 0.920289       0.662465 0.067290 120457   0.128179
logit       CASE_COURT valid 0.920785       0.665713 0.067026 120457   0.128179
logit CASE_COURT_JUDGE valid 0.923637       0.677013 0.065788 120457   0.128179
logit CASE_PLUS_GENDER valid 0.921347       0.667988 0.066639 120457   0.128179
lasso             CASE valid 0.920337       0.662595 0.067270 120457   0.128179
lasso       CASE_COURT valid 0.920855       0.665698 0.067029 120457   0.128179
lasso CASE_COURT_JUDGE valid 0.923789       0.677448 0.065698 120457   0.128179
lasso CASE_PLUS_GENDER valid 0.921149       0.667428 0.066701 120457   0.128179
 tree             CASE valid 0.922505       0.655781 0.067615 120457   0.128179
 tree       CASE_COURT valid 0.922019       0.656

feature_set      CASE                                             CASE_COURT                                             CASE_COURT_JUDGE                                            
metric        roc_auc avg_precision     brier         n base_rate    roc_auc avg_precision     brier         n base_rate          roc_auc avg_precision     brier         n base_rate
model                                                                                                                                                                                
lasso        0.920337      0.662595  0.067270  120457.0  0.128179   0.920855      0.665698  0.067029  120457.0  0.128179         0.923789      0.677448  0.065698  120457.0  0.128179
logit        0.920289      0.662465  0.067290  120457.0  0.128179   0.920785      0.665713  0.067026  120457.0  0.128179         0.923637      0.677013  0.065788  120457.0  0.128179
rule         0.846492      0.478021  0.080927  120457.0  0.128179        NaN           NaN

## Supplementary gender comparison

The following diagnostic is separate from the main nested comparison. It measures incremental predictive signal for historical decisions on the same validation dockets. Different hyperparameters may be selected for each feature set. It does not identify discriminatory treatment or establish that observed case information fully explains group disparities. We retain the BD-04 paired-bootstrap comparison for logistic regression.

In [5]:
valid_metrics = summary.loc[summary["split"].eq("valid")].set_index(["model", "feature_set"])
gender_differences = []
for model in ["logit", "lasso", "tree"]:
    before = valid_metrics.loc[(model, "CASE")]
    after = valid_metrics.loc[(model, "CASE_PLUS_GENDER")]
    gender_differences.append({"model": model,
        **{f"delta_{m}": float(after[m] - before[m]) for m in ["roc_auc", "avg_precision", "brier"]}})
gender_differences = pd.DataFrame(gender_differences)
gender_differences.to_csv(config.REPORTS / "bd05_gender_increment_valid.csv", index=False)
print(gender_differences.to_string(index=False, float_format=lambda v: f"{v:+.6f}"))
base, plus = [
    pred_df.loc[pred_df["model"].eq("logit") & pred_df["feature_set"].eq(fs) & pred_df["split"].eq("valid")]
    .set_index(config.ID_COL).sort_index()
    for fs in ["CASE", "CASE_PLUS_GENDER"]
]
assert base.index.equals(plus.index)
assert base["y_true"].equals(plus["y_true"])

model  delta_roc_auc  delta_avg_precision  delta_brier
logit      +0.001058            +0.005523    -0.000651
lasso      +0.000811            +0.004833    -0.000569
 tree      -0.000357            +0.000193    +0.000088


### Uncertainty of the logistic gender comparison

Paired docket bootstrap over the validation year uses the configured resample count and seed. Positive AUC/AP differences favor CASE_PLUS_GENDER. This interval is conditional on the fitted models and does not account for retraining or judge-level dependence.

In [6]:
from sklearn.metrics import average_precision_score, roc_auc_score
from src.evaluate import paired_bootstrap_diff

gender_rows = []
for name, fn in [("roc_auc", roc_auc_score), ("avg_precision", average_precision_score)]:
    est, lo, hi = paired_bootstrap_diff(base["y_true"], base["y_prob"], plus["y_prob"], fn)
    gender_rows.append({"metric": name, "plus_gender_minus_case": est, "ci_2.5": lo, "ci_97.5": hi})
gender_ci = pd.DataFrame(gender_rows)
gender_ci.to_csv(config.REPORTS / "bd05_logit_gender_ci_valid.csv", index=False)
print(gender_ci.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

       metric  plus_gender_minus_case  ci_2.5  ci_97.5
      roc_auc                 +0.0011 +0.0007  +0.0014
avg_precision                 +0.0055 +0.0044  +0.0067


## Interpretation and handoff

Use the validation table for model comparison. AUC measures ranking, not the percentage of correct decisions; AP is interpreted against the validation positive-label rate; Brier describes probability error. The rule provides a simple charge-only reference. Main feature-set differences are predictive associations, not causal court or judge effects. About 30% of validation cases have an unseen judge under the training-year map, so BD-06 must also show the subset with individually coded judges.

BD-05 supplies local predictions_baseline.parquet and all twelve fitted models. Individual predictions and joblib files must not be committed. Only aggregate CSVs and this notebook are review candidates. BD-06 will combine the ensemble predictions, add paired intervals for nested feature gains and calibration, and support the group model decision. Thresholds and fairness remain with C. The final-test path stays disabled pending the separate freeze and BD-07.

Prior-record/pending-case reference dates remain undocumented (data/README.md, D14). This work predicts historical decisions, not actual detention, future offending or a policy recommendation. No test-year outcome or model performance is calculated.

In [7]:
print("BD-05 complete: nine main fits, three supplementary fits, one charge rule.")
print("2024 raw source not read; no test predictions produced; final-test gate remains closed.")

BD-05 complete: nine main fits, three supplementary fits, one charge rule.
2024 raw source not read; no test predictions produced; final-test gate remains closed.
